# Translate Title and Abstracts of non-english publications into english
We now have a Dataframe with all the queried publications and their references. We want to translate all the titles and abstracts to english, to be able to perform further analysis on the text. We use the [OpenAI GPT-4o](https://openai.com/index/hello-gpt-4o) for this task, since it is scalable and gives high quality results. We tried out various other translation APIs or Open Source solutions, like [Cloud Translation API ](https://cloud.google.com/translate?hl=en), [EasyNMT](https://github.com/UKPLab/EasyNMT) and [Huggingface Transformer](https://huggingface.co/docs/transformers/model_doc/marian), but none of them can match the quality, speed (or price for the paid services) of the GPT-4o Model. In the future, we will switch to an Open Source alternative, which unfortunately does not currently meet our required quality, which is true for STEM texts.

## Imports

In [1]:
import pandas as pd
import os.path
import fasttext
import os
import openai
from dotenv import load_dotenv
import spacy
import numpy as np
from semanticlayertools.cleaning.text import tokenize

Consider loading a larger language model. Falling back to using small english.


c:\Users\rapha\anaconda3\envs\ADS_env\lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.2). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


## Load Data

In [2]:
# Load DataFrame
df = pd.read_json('2_data/queried_publications.json', lines=True)
dfr = pd.read_json('2_data/queried_references.json', lines=True)

## Language Detection
Before we start to translate we want to detect the language in each relevant cell of our dataframe to reduce time and cost before sending any text to a translation API. The detection could also be done using the Google Translation API or GPT-4o, but if we check every entry, if it is english this will get expensive very fast. Using fastText for Language detection is a much faster and free method to do that. We will add the detected languages to new colums, so that we will only translate those cells, which are not in english.

In [3]:
# Load the pre-trained Language Identification model
model = fasttext.load_model('3_models/lid.176.bin')

# Function to predict language of a text
def predict_language(text):
    return model.predict(text)[0][0].replace("__label__", "")

# Function to apply language detection to a column
def detect_language_in_column(df, column_name):
    df[column_name] = df[column_name].fillna('')
    df[column_name + '_lang'] = df[column_name].apply(predict_language)

# Detect language in specified columns
detect_language_in_column(df, 'Title')
detect_language_in_column(df, 'Abstract')

# write the resulting DataFrame to a JSON-File.
df.to_json('2_data/queried_publications_lang.json', orient='records', lines=True)

We'll do the same for the references.

In [4]:
# Detect language in specified columns
detect_language_in_column(dfr, 'Title')
detect_language_in_column(dfr, 'Abstract')

# write the resulting DataFrame to a JSON-File.
dfr.to_json('2_data/queried_references_lang.json', orient='records', lines=True)

We then count the number of titles and abstracts that are not in english, to see how much we actually have to translate.

In [5]:
# Count the number of rows in a dataframe where the specified language column is not English ('en').
def count_filtered_rows(df, lang_column):
    df_filtered = df[df[lang_column] != 'en']
    return len(df_filtered)

# For queried_publications
title_filtered_count = count_filtered_rows(df, 'Title_lang')
abstract_filtered_count = count_filtered_rows(df, 'Abstract_lang')

print(f"Number of non-English entries in 'Title' column of queried_publications: {title_filtered_count}")
print(f"Number of non-English entries in 'Abstract' column of queried_publications: {abstract_filtered_count}")

# For queried_references
title_filtered_count_r = count_filtered_rows(dfr, 'Title_lang')
abstract_filtered_count_r = count_filtered_rows(dfr, 'Abstract_lang')

print(f"Number of non-English entries in 'Title' column of queried_references: {title_filtered_count_r}")
print(f"Number of non-English entries in 'Abstract' column of queried_references: {abstract_filtered_count_r}")

Number of non-English entries in 'Title' column of queried_publications: 4303
Number of non-English entries in 'Abstract' column of queried_publications: 772
Number of non-English entries in 'Title' column of queried_references: 8662
Number of non-English entries in 'Abstract' column of queried_references: 1188


In the whole of our corpus we only have 364 abstract entries which are not in english. That means we only have to translate those. But before we start to translate, lets look at how many sentences, words and tokens we have in this reduced dataset to potentially translate. This is useful to estimate the cost of the translation, which can get expensive for large datasets.

## Translation with OpenAI GPT-4o
Now we turn to the actual translation. Before we start to acutally send calls to the API, we roughly estimate the costs, so that we it is clear beforehand, what scale to expect. 

In [6]:
import os
import pandas as pd
import tiktoken  # OpenAI's tokenizer package
from dotenv import load_dotenv

# Load environment variables from .env file to get the OpenAI API key
load_dotenv()

# Model pricing configuration
model_config = {
    "gpt-4o": {"input_price": 0.0025, "output_price": 0.01},
    "gpt-4o-2024-08-06": {"input_price": 0.0025, "output_price": 0.01},
    "gpt-4o-mini": {"input_price": 0.00015, "output_price": 0.0006},
}

def estimate_cost(i_tokens, o_tokens, model_name):
    """Estimate and print the cost of processing based on input and output token counts."""
    input_price = model_config[model_name]["input_price"]
    output_price = model_config[model_name]["output_price"]
    i_cost = (i_tokens / 1000) * input_price
    o_cost = (o_tokens / 1000) * output_price
    total_cost = i_cost + o_cost
    print(f"Model: {model_name}\nToken Usage\nPrompt: {i_tokens} tokens\nCompletion: {o_tokens} tokens\nCost estimation: ${round(total_cost, 5)}")
    return round(total_cost, 5)

def tokenize_text(text, model_name):
    """Tokenize the text using OpenAI's tokenizer and return the token count."""
    enc = tiktoken.encoding_for_model(model_name)
    tokens = enc.encode(text)
    return len(tokens)

def calculate_estimated_cost(df, model_name='gpt-4o'):
    """Calculate and estimate the cost for translating the 'Title' and 'Abstract' columns."""
    total_i_tokens = sum(df[df['Title_lang'] != 'en']['Title'].apply(lambda x: tokenize_text(x, model_name))) + \
                     sum(df[df['Abstract_lang'] != 'en']['Abstract'].apply(lambda x: tokenize_text(x, model_name)))
    
    # Estimate the cost
    estimated_cost = estimate_cost(total_i_tokens, total_i_tokens, model_name)  # Assuming similar output token count
    return estimated_cost

estimated_cost = calculate_estimated_cost(df, model_name='gpt-4o')


Model: gpt-4o
Token Usage
Prompt: 275698 tokens
Completion: 275698 tokens
Cost estimation: $3.44623


In [13]:
# Load environment variables from .env file to get the OpenAI API key
from openai import OpenAI

load_dotenv()

# Set the OpenAI API key
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

# Global variables to keep track of token usage
total_i_tokens = 0
total_o_tokens = 0

# Model pricing configuration
model_config = {
    "gpt-4o": {"input_price": 0.0025, "output_price": 0.01},
    "gpt-4o-2024-08-06": {"input_price": 0.0025, "output_price": 0.01},
    "gpt-4o-mini": {"input_price": 0.00015, "output_price": 0.0006},
}

def estimate_cost(i_tokens, o_tokens, model_name):
    """Estimate and print the cost of processing based on input and output token counts."""
    input_price = model_config[model_name]["input_price"]
    output_price = model_config[model_name]["output_price"]
    i_cost = (i_tokens / 1000) * input_price
    o_cost = (o_tokens / 1000) * output_price
    total_cost = i_cost + o_cost
    print(f"Model: {model_name}\nToken Usage\nPrompt: {i_tokens} tokens\nCompletion: {o_tokens} tokens\nCost estimation: ${round(total_cost, 5)}")

def translate_text(text, target_lang='en', model_name='gpt-4o'):
    """
    Translate a given text to the target language using OpenAI's GPT-4 model.
    
    :param text: The input text to be translated.
    :param target_lang: The target language code for translation (default is 'en' for English).
    :param model_name: The name of the OpenAI model to use for translation.
    :return: The translated text.
    """
    global total_i_tokens, total_o_tokens

    try:
        # Create the translation prompt for scientific accuracy
        messages = [
            {
                "role": "system",
                "content": "You are a highly accurate translator specializing in scientific and technical texts. Only translate the text. Do not comment or provide additional information."
            },
            {
                "role": "user",
                "content": f"Translate the following physics text to {target_lang}:\n\n{text}"
            }
        ]
        
        # Call OpenAI API for translation
        response = client.chat.completions.create(
            messages=messages,
            model=model_name,
            max_tokens=2048,
            temperature=0  # Low temperature for more accurate translations
        )

        # Extracting the token usage information from the response
        i_tokens = response.usage.prompt_tokens
        o_tokens = response.usage.completion_tokens

        # Add the token counts to the global variables
        total_i_tokens += i_tokens
        total_o_tokens += o_tokens

        # Return the translated text
        return response.choices[0].message.content.strip()
    except Exception as e:
        # Return error message if any exception occurs
        return f"Error: {str(e)}"

def translate_dataframe_column(df, column, lang_col, target_lang='en', model_name='gpt-4o'):
    """
    Translate a specific column in a pandas DataFrame using OpenAI's GPT-4 model.
    Translation will only occur if the value in lang_col is not 'en' (English).
    
    :param df: The input pandas DataFrame.
    :param column: The name of the column to be translated.
    :param lang_col: The name of the column that holds the language code for each row.
    :param target_lang: The target language code for translation.
    :param model_name: The name of the OpenAI model to use for translation.
    :return: The DataFrame with the translated column and a list of failed translations.
    """
    failed_translations = []

    def translate_cell(row):
        """
        Translate a cell in the DataFrame if it is not in the target language.
        
        :param row: The row of the DataFrame being processed.
        :return: The translated cell value or the original value in case of an error.
        """
        try:
            if pd.isnull(row[column]) or row[lang_col] == 'en':
                return row[column]
            else:
                translated_text = translate_text(row[column], target_lang, model_name)
                if "Error:" in translated_text:
                    failed_translations.append((row.name, translated_text))
                    return row[column]  # Return the original value in case of error
                else:
                    return translated_text
        except Exception as e:
            failed_translations.append((row.name, str(e)))
            return row[column]  # Return the original value

    # Apply the translation function to the specified column
    df[column + "_" + target_lang] = df.apply(translate_cell, axis=1)
    
    return df, failed_translations

def translate_json_file(json_file, model_name='gpt-4o'):
    """
    Translate the 'Title' and 'Abstract' columns of a JSON file into English using OpenAI's GPT-4 model.
    
    :param json_file: The path to the JSON file containing the data.
    :param model_name: The name of the OpenAI model to use for translation.
    """
    # Read the JSON file into a pandas DataFrame
    df = pd.read_json(json_file, lines=True)

    # Apply the function to the 'Title' column and translate into English
    df_translated, failed_title = translate_dataframe_column(df, "Title", "Title_lang", "en", model_name)
    print(f"Failed translations (Title): {len(failed_title)} out of {len(df[df['Title_lang'] != 'en'])}")
    print("Failed translations details (Title):", failed_title)

    # Apply the function to the 'Abstract' column and translate into English
    df_translated, failed_abstract = translate_dataframe_column(df_translated, "Abstract", "Abstract_lang", "en", model_name)
    print(f"Failed translations (Abstract): {len(failed_abstract)} out of {len(df[df['Abstract_lang'] != 'en'])}")
    print("Failed translations details (Abstract):", failed_abstract)

    # Save the translated DataFrame to a new JSON file
    base_name, ext = os.path.splitext(json_file)
    translated_file = f"{base_name}_translated_openai{ext}"
    df_translated.to_json(translated_file, orient='records', lines=True)
    
    # Estimate and print the cost
    estimate_cost(total_i_tokens, total_o_tokens, model_name)

In [15]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
from openai import OpenAI
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from .env file to get the OpenAI API key
load_dotenv()

# Set the OpenAI API key
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

# Global variables to keep track of token usage
total_i_tokens = 0
total_o_tokens = 0

# Model pricing configuration
model_config = {
    "gpt-4o": {"input_price": 0.0025, "output_price": 0.01},
    "gpt-4o-2024-08-06": {"input_price": 0.0025, "output_price": 0.01},
    "gpt-4o-mini": {"input_price": 0.00015, "output_price": 0.0006},
}

def estimate_cost(i_tokens, o_tokens, model_name):
    """Estimate and print the cost of processing based on input and output token counts."""
    input_price = model_config[model_name]["input_price"]
    output_price = model_config[model_name]["output_price"]
    i_cost = (i_tokens / 1000) * input_price
    o_cost = (o_tokens / 1000) * output_price
    total_cost = i_cost + o_cost
    print(f"Model: {model_name}\nToken Usage\nPrompt: {i_tokens} tokens\nCompletion: {o_tokens} tokens\nCost estimation: ${round(total_cost, 5)}")

def translate_text(text, target_lang='en', model_name='gpt-4o'):
    """Translate a given text to the target language using OpenAI's GPT-4 model."""
    global total_i_tokens, total_o_tokens

    try:
        # Create the translation prompt for scientific accuracy
        messages = [
            {
                "role": "system",
                "content": "You are a highly accurate translator specializing in scientific and technical texts. Only translate the text. Do not comment or provide additional information."
            },
            {
                "role": "user",
                "content": f"Translate the following physics text to {target_lang}:\n\n{text}"
            }
        ]
        
        # Call OpenAI API for translation
        response = client.chat.completions.create(
            messages=messages,
            model=model_name,
            max_tokens=2048,
            temperature=0  # Low temperature for more accurate translations
        )

        # Extracting the token usage information from the response
        i_tokens = response.usage.prompt_tokens
        o_tokens = response.usage.completion_tokens

        # Add the token counts to the global variables
        total_i_tokens += i_tokens
        total_o_tokens += o_tokens

        # Return the translated text
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"

def batch_translate_texts(texts, target_lang='en', model_name='gpt-4o'):
    """Batch translate a list of texts."""
    try:
        messages = [
            {
                "role": "system",
                "content": "You are a highly accurate translator specializing in scientific and technical texts. Only translate the text. Do not comment or provide additional information."
            }
        ] + [{"role": "user", "content": f"Translate the following text to {target_lang}:\n\n{text}"} for text in texts]

        response = client.chat.completions.create(
            messages=messages,
            model=model_name,
            max_tokens=2048,
            temperature=0  # Low temperature for more accurate translations
        )

        # Extracting the token usage information from the response
        i_tokens = response.usage.prompt_tokens
        o_tokens = response.usage.completion_tokens

        global total_i_tokens, total_o_tokens
        total_i_tokens += i_tokens
        total_o_tokens += o_tokens

        return [choice.message.content.strip() for choice in response.choices]
    except Exception as e:
        return [f"Error: {str(e)}" for _ in texts]

def parallel_translate_dataframe_column(df, column, lang_col, target_lang='en', model_name='gpt-4o', max_workers=5):
    """Parallelize translation of a specific column in a pandas DataFrame."""
    failed_translations = []

    def translate_row(row):
        """Translate a row's column if it's not in the target language."""
        if pd.isnull(row[column]) or row[lang_col] == 'en':
            return row[column]
        else:
            translated_text = translate_text(row[column], target_lang, model_name)
            if "Error:" in translated_text:
                failed_translations.append((row.name, translated_text))
            return translated_text

    # Use ThreadPoolExecutor for parallel processing
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(translate_row, row): row.name for _, row in df.iterrows()}

        for future in as_completed(futures):
            row_idx = futures[future]
            try:
                df.at[row_idx, column + "_" + target_lang] = future.result()
            except Exception as exc:
                failed_translations.append((row_idx, str(exc)))

    return df, failed_translations

def translate_json_file(json_file, model_name='gpt-4o'):
    """Translate the 'Title' and 'Abstract' columns of a JSON file into English using OpenAI's GPT-4 model."""
    df = pd.read_json(json_file, lines=True)

    # Translate the 'Title' column in parallel
    df_translated, failed_title = parallel_translate_dataframe_column(df, "Title", "Title_lang", "en", model_name)
    print(f"Failed translations (Title): {len(failed_title)} out of {len(df[df['Title_lang'] != 'en'])}")
    print("Failed translations details (Title):", failed_title)

    # Translate the 'Abstract' column in parallel
    df_translated, failed_abstract = parallel_translate_dataframe_column(df_translated, "Abstract", "Abstract_lang", "en", model_name)
    print(f"Failed translations (Abstract): {len(failed_abstract)} out of {len(df[df['Abstract_lang'] != 'en'])}")
    print("Failed translations details (Abstract):", failed_abstract)

    # Save the translated DataFrame to a new JSON file
    base_name, ext = os.path.splitext(json_file)
    translated_file = f"{base_name}_translated_openai{ext}"
    df_translated.to_json(translated_file, orient='records', lines=True)

    # Estimate and print the cost
    estimate_cost(total_i_tokens, total_o_tokens, model_name)


Let's apply the translation to the non-english titles and abstracts.

In [8]:
translate_json_file('2_data/queried_publications_lang.json', model_name="gpt-4o")

Failed translations (Title): 0 out of 4303
Failed translations details (Title): []
Failed translations (Abstract): 0 out of 772
Failed translations details (Abstract): []
Model: gpt-4o
Token Usage
Prompt: 504073 tokens
Completion: 184934 tokens
Cost estimation: $3.10952


In [16]:
translate_json_file('2_data/queried_references_lang.json', model_name="gpt-4o")

Failed translations (Title): 0 out of 8662
Failed translations details (Title): []
Failed translations (Abstract): 0 out of 1188
Failed translations details (Abstract): []
Model: gpt-4o
Token Usage
Prompt: 956966 tokens
Completion: 342438 tokens
Cost estimation: $5.81679


We have no failed translations and have paid a total sum of xxx $. 

## Tokenize Titles and Abstracts

The last thing we will do to our data is to tokenize the combined text of the translated Title and Abstract column of our queried publications. For that we will use the [`semanticlayertools`](https://semanticlayertools.readthedocs.io)[^1] cleaning routine.

[^1]: The developement of [`semanticlayertools`](https://semanticlayertools.readthedocs.io) is part of the research project [`ModelSEN`](https://modelsen.mpiwg-berlin.mpg.de) and is a collection of tools to build, cluster, analyse and visualize social, semiotic or semantic layers from text corpora.

In [17]:
# Load spaCy model
nlp = spacy.load("en_core_web_lg")

# Define function to process DataFrame
def process_dataframe(json_file, output_file):
    # Read the JSON file
    df = pd.read_json(json_file, lines=True)

    # Combine 'Title' and 'Abstract' into 'full_text'
    df['full_text'] = df['Title_en'].fillna('').astype(str) + ". " + df['Abstract_en'].fillna('').astype(str)

    # Tokenize 'full_text'
    df['tokens'] = df['full_text'].apply(lambda x: tokenize(
        x, languageModel=nlp, ngramRange=(1, 1), 
        excludeStopWords=True, excludePunctuation=True, 
        excludeNumerical=True, excludeNonAlphabetic=True, 
        tokenMinLength=3
    ))

    # Save the processed DataFrame to a new JSON file
    df.to_json(output_file, orient="records", lines=True)

We apply the tokenization to the translated titles and abstracts and save the results in new files.

In [18]:
# Process queried_publications and queried_references
# process_dataframe('2_data/queried_publications_lang_translated_openai.json', '2_data/queried_publications_lang_translated_openai_tokenized.json')
process_dataframe('2_data/queried_references_lang_translated_openai.json', '2_data/queried_references_lang_translated_openai_tokenized.json')

Let's look at our new dataframe with the translated and tokenized titles and abstracts.

In [19]:
#| column: page
#| label: translated and tokeized

# Show the first 5 rows of the translated DataFrame
df_translated = pd.read_json('2_data/queried_references_lang_translated_openai_tokenized.json', lines=True)
df_translated

,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,...,DOI,Affiliation,Category,Citation Count,Title_lang,Abstract_lang,Title_en,Abstract_en,full_text,tokens
0,1998AJ....116.1009R,"Riess AG, Filippenko AV, Challis P, Clocchiatt...",Observational Evidence from Supernovae for an ...,1998,The Astronomical Journal,AJ,3,116,1009,1038,...,10.1086/300499,"AA(Department of Astronomy, University of Cali...",article,15988.0,en,en,Observational Evidence from Supernovae for an ...,We present spectral and photometric observatio...,Observational Evidence from Supernovae for an ...,"[observational, evidence, supernovae, accelera..."
1,1992nrfa.book.....P,"Press WH, Teukolsky SA, Vetterling WT, Flanner...",Numerical recipes in FORTRAN. The art of scien...,1992,Cambridge: University Press,nrfa.book,,,,,...,,AA(-); AB(-); AC(-); AD(-),book,13563.0,en,en,Numerical recipes in FORTRAN. The art of scien...,http://www.nr.com,Numerical recipes in FORTRAN. The art of scien...,"[numerical, recipe, fortran, art, scientific, ..."
2,1998AdTMP...2..231M,Maldacena JM,The Large N Limit of Superconformal Field Theo...,1998,Advances in Theoretical and Mathematical Physics,AdTMP,2,2,231,,...,10.4310/ATMP.1998.v2.n2.a1,AA(-),article,11299.0,en,en,The Large N Limit of Superconformal Field Theo...,We show that the large N limit of certain conf...,The Large N Limit of Superconformal Field Theo...,"[large, limit, superconformal, field, theories..."
3,1998AdTMP...2..253W,Witten E,Anti-de Sitter space and holography,1998,Advances in Theoretical and Mathematical Physics,AdTMP,,2,253,291,...,10.48550/arXiv.hep,AA(-),article,11661.0,en,en,Anti-de Sitter space and holography,"Recently, it has been proposed by Maldacena th...","Anti-de Sitter space and holography. Recently,...","[anti, sitter, space, holography, recently, pr..."
4,1935PhRv...47..777E,"Einstein A, Podolsky B, Rosen N",Can Quantum-Mechanical Description of Physical...,1935,Physical Review,PhRv,10,47,777,780,...,10.1103/PhysRev.47.777,"AA(Institute for Advanced Study, Princeton, Ne...",article,9244.0,en,en,Can Quantum-Mechanical Description of Physical...,In a complete theory there is an element corre...,Can Quantum-Mechanical Description of Physical...,"[quantum, mechanical, description, physical, r..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
509244,1994MatL...20..231A,"Abd El Wahed MG, Metwally SM",Electrical conductivity of amino-hydroxy-napht...,1994,Materials Letters,MatL,3,20,231,236,...,10.1016/0167,"AA(Faculty of Science, Zagazig University, Zag...",article,1.0,en,en,Electrical conductivity of amino-hydroxy-napht...,The influence of complexation with some divale...,Electrical conductivity of amino-hydroxy-napht...,"[electrical, conductivity, amino, hydroxy, nap..."
509245,1991IAUS..144..369P,Pohl M,The Influence of Extended Source Distributions...,1991,The Interstellar Disk-Halo Connection in Galaxies,IAUS,,144,369,,...,,AA(-),inproceedings,1.0,en,en,The Influence of Extended Source Distributions...,,The Influence of Extended Source Distributions...,"[influence, extended, source, distribution, co..."
509246,1971NCimA...5..513B,"Benayoun M, Leruste P",Local models for many-channel low-energy $bar ...,1971,Nuovo Cimento A Serie,NCimA,4,5,513,526,...,10.1007/BF02734562,AA(-); AB(-),article,1.0,en,en,Local models for many-channel low-energy $bar ...,,Local models for many-channel low-energy $bar ...,"[local, model, channel, low, energy, bar, scat..."
509247,1987RaF....30..917V,"Vostokov AV, Dugin NA, Miller ME, Syreishchiko...",Absolute measurements of the flux density of i...,1987,Radiofizika,RaF,7,30,917,919,...,,AA(Nauchno-Issledovatel'skii Radiofizicheskii ...,article,1.0,en,en,Absolute measurements of the flux density of i...,Absolute measurements of several discrete radi...,Absolute measurements of the flux density of i...,"[absolute, measurement, flux, density, intense..."
